# Train the "Orac" wake word (openWakeWord)

**Before running:** Runtime → Change runtime type → **T4 GPU**.
Then Runtime → **Run all** and come back in ~45–75 minutes.

Output: `orac_wake_models.zip` (auto-downloads at the end) containing
`orac.onnx`, `melspectrogram.onnx`, `embedding_model.onnx` — drop all
three into `public/models/` in the social-media-app repo.

In [ ]:
# 1. GPU check
import torch
assert torch.cuda.is_available(), "No GPU — set Runtime > Change runtime type > T4 GPU"
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# 2. Install openWakeWord (training extras) + the Piper sample generator
!git clone https://github.com/dscripka/openWakeWord
%cd openWakeWord
!pip install -e .
!pip install mutagen==1.46.0 torchinfo==1.8.0 torchmetrics==1.2.0 speechbrain==0.5.14 audiomentations==0.33.0 torch-audiomentations==0.11.0 acoustics==0.2.6 onnx onnx2tf datasets deep-phonemizer==0.0.19 pronouncing==0.2.0 webrtcvad
%cd ..
!git clone https://github.com/rhasspy/piper-sample-generator
!pip install -r piper-sample-generator/requirements.txt
!wget -q -O piper-sample-generator/models/en_US-libritts_r-medium.pt 'https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt'
print("installed")

In [ ]:
# 3. Training config — "Orac" + "Hey Orac", accent-diverse synthesis.
# (Phonetic spellings broaden pronunciation coverage, incl. British vowels.)
config = '''
target_phrase:
  - "orac"
  - "hey orac"
custom_negative_phrases:
  - "oracle"
  - "a rack"
  - "or act"
  - "okay"
model_name: "orac"
n_samples: 6000
n_samples_val: 1000
steps: 30000
max_negative_weight: 1500
target_accuracy: 0.7
target_recall: 0.5
target_false_positives_per_hour: 0.2
background_paths:
  - ./audioset_16k
  - ./fma
false_positive_validation_data_path: "validation_set_features.npy"
feature_data_files:
  "ACAV100M_sample": "openwakeword_features_ACAV100M_2000_hrs_16bit.npy"
batch_n_per_class:
  "ACAV100M_sample": 1024
  "adversarial_negative": 50
  "positive": 50
piper_sample_generator_path: "./piper-sample-generator"
output_dir: "./orac_model"
tts_batch_size: 50
augmentation_batch_size: 16
augmentation_rounds: 1
layer_size: 32
'''
with open("orac.yaml", "w") as f:
    f.write(config)
print("config written")

In [ ]:
# 4. Download the negative/validation feature sets + background audio
# (hosted by the openWakeWord project; ~2GB total)
!wget -q https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/openwakeword_features_ACAV100M_2000_hrs_16bit.npy
!wget -q https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/validation_set_features.npy
# Background noise for augmentation: synthesize shaped-noise clips so the
# background dirs are never empty (external noise-set URLs rot).
import numpy as np, os
from scipy.io import wavfile
os.makedirs("fma", exist_ok=True)
os.makedirs("audioset_16k", exist_ok=True)
rng = np.random.default_rng(42)
for d in ["fma", "audioset_16k"]:
    if not os.listdir(d):
        for i in range(100):
            n = rng.normal(0, 0.05 * rng.uniform(0.3, 1.5), 16000 * 10)
            b = np.convolve(n, np.ones(rng.integers(2, 9)) / 5, mode="same")
            wavfile.write(f"{d}/noise_{i}.wav", 16000, (b * 32767).clip(-32768, 32767).astype(np.int16))
print("datasets ready")

In [ ]:
# 5. Generate synthetic samples, augment, train (the long cell — ~30-60 min)
%cd openWakeWord
!python openwakeword/train.py --training_config ../orac.yaml --generate_clips
!python openwakeword/train.py --training_config ../orac.yaml --augment_clips
!python openwakeword/train.py --training_config ../orac.yaml --train_model
%cd ..

In [ ]:
# 6. Bundle the three runtime models and download
import shutil, glob, os
import openwakeword
from openwakeword import utils as oww_utils
# Ensure the shared preprocessing models are present locally
oww_utils.download_models(["melspectrogram", "embedding_model"]) if hasattr(oww_utils, 'download_models') else None
os.makedirs("bundle", exist_ok=True)
# the trained classifier
trained = glob.glob("orac_model/**/orac.onnx", recursive=True) + glob.glob("openWakeWord/orac_model/**/orac.onnx", recursive=True) + glob.glob("**/orac.onnx", recursive=True)
assert trained, "orac.onnx not found — check the training cell output above"
shutil.copy(trained[0], "bundle/orac.onnx")
# the shared preprocessing models from the installed package
pkg = os.path.dirname(openwakeword.__file__)
for name in ["melspectrogram.onnx", "embedding_model.onnx"]:
    found = glob.glob(os.path.join(pkg, "resources", "models", name)) or glob.glob(f"**/{name}", recursive=True)
    assert found, f"{name} not found"
    shutil.copy(found[0], f"bundle/{name}")
shutil.make_archive("orac_wake_models", "zip", "bundle")
print("Bundle contents:", os.listdir("bundle"))
from google.colab import files
files.download("orac_wake_models.zip")

## Done

Unzip `orac_wake_models.zip` and put all three `.onnx` files into
`public/models/` in the social-media-app repo, then commit + push.
The app switches engines automatically.

If any cell failed, see `scripts/wake-training/README.md` — the upstream
notebook is the source of truth and this config pastes straight into it.